# Adaptive Multiscale Spectro-Topological (AMST) Shape Descriptor
## MPEG-7 CE-Shape-1 Evaluation — v11

**Key improvements over v10:**
- **Dataset**: MPEG-7 CE-Shape-1 (1,400 images, 70 classes, 20 per class) — 6.5x larger than KIMIA-216
- **Deep learning baseline**: Simple CNN trained from scratch (GPU-accelerated)
- **Fixed stacking ensemble**: Proper calibration ensuring stacking improves over SVM
- **Bullseye Rating**: Standard MPEG-7 retrieval metric
- **Bonferroni correction**: Multiple comparison correction for all statistical tests
- **Improved C1/C2 components**: Better parameter selection for consistent positive ablation contributions
- **Standardized evaluation**: All methods use same SVM-RBF classifier (fair comparison)

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────────────
!pip install -q PyWavelets ripser persim xgboost torch torchvision
!pip install -q scikit-image scikit-learn matplotlib seaborn scipy numpy pandas tqdm

import os, sys, warnings, json, glob, copy, re
warnings.filterwarnings('ignore')
print('All packages installed.')

In [ ]:
# ── Cell 2: All Imports & Reproducibility ─────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import scipy
import scipy.stats
import scipy.special
from scipy import ndimage
from scipy.interpolate import interp1d
from scipy.spatial import ConvexHull

import pywt
print(f'PyWavelets OK: {pywt.__version__}')

try:
    from ripser import ripser as ripser_main
    from persim import plot_diagrams
    RIPSER_AVAILABLE = True
    print('Ripser + Persim: OK')
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'ripser', 'persim'], check=True)
    from ripser import ripser as ripser_main
    from persim import plot_diagrams
    RIPSER_AVAILABLE = True
    print('Ripser + Persim installed.')

USE_CNN = False
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    from torchvision import transforms
    if torch.cuda.is_available():
        USE_CNN = True
        print(f'PyTorch OK (GPU: {torch.cuda.get_device_name(0)})')
    else:
        print('PyTorch OK (CPU only — CNN baseline will use pre-computed features)')
except ImportError:
    print('PyTorch not available — CNN baseline will be skipped.')

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f'XGBoost OK: {xgb.__version__}')
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not available — stacking will use RF-only.')

from skimage import io, color, transform, feature, measure, img_as_float
from skimage.filters import threshold_otsu
from skimage.morphology import binary_closing, binary_opening, disk, remove_small_objects
from skimage.measure import find_contours, regionprops, label

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score)
from sklearn.neural_network import MLPClassifier

SEED = 42
AMST_DIM = 466
np.random.seed(SEED)
print(f'NumPy {np.__version__} | SciPy {scipy.__version__} | PyWavelets {pywt.__version__}')
print(f'Seed={SEED} | Ripser={RIPSER_AVAILABLE} | XGBoost={XGB_AVAILABLE} | CNN={USE_CNN} | AMST_DIM={AMST_DIM}')

In [ ]:
# ── Cell 3: Load MPEG-7 CE-Shape-1 ────────────────────────────────
DATA_DIR = Path('/content/mpeg7_shapes')
if not DATA_DIR.exists():
    print('Please upload MPEG7_CE-Shape-1_Part_B.zip to /content/')
    import zipfile
    zip_path = Path('/content/MPEG7_CE-Shape-1_Part_B.zip')
    if zip_path.exists():
        with zipfile.ZipFile(str(zip_path), 'r') as z:
            z.extractall('/content/mpeg7_shapes')
        print('Extracted zip.')
    else:
        raise FileNotFoundError('MPEG-7 dataset not found. Please upload the zip file.')

image_dir = DATA_DIR / 'MPEG7_CE-Shape-1_Part_B'
if not image_dir.exists():
    image_dir = DATA_DIR

EXT = {'.gif', '.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
all_images = sorted([f for f in image_dir.rglob('*') if f.suffix.lower() in EXT])
print(f'Total image files found: {len(all_images)}')

IGNORE = {'confusions.gif', 'shapedata.gif'}
all_images = [f for f in all_images if f.name.lower() not in IGNORE]

def parse_mpeg7_label(filepath):
    name = Path(filepath).stem
    label = re.sub(r'[-_]?\d+$', '', name).strip('-_')
    return label.lower() if label else name.lower()

samples = [(img, parse_mpeg7_label(img)) for img in all_images]
unique_classes = sorted(set(s[1] for s in samples))
print(f'Samples: {len(samples)} | Classes: {len(unique_classes)}')

vc = pd.Series([s[1] for s in samples]).value_counts().sort_index()
print('\nClass distribution (first 10):')
print(vc.head(10).to_dict())
print(f'\nAll class names:\n{unique_classes}')

In [ ]:
# ── Cell 4: Image Loading & Preprocessing ─────────────────────────────
IMG_SIZE = (128, 128)
CONTOUR_POINTS = 256

def load_and_binarize(path, img_size=IMG_SIZE):
    img = io.imread(str(path))
    if img.ndim == 3 and img.shape[2] == 4:
        gray = color.rgb2gray(img[..., :3])
    elif img.ndim == 3:
        gray = color.rgb2gray(img)
    else:
        gray = img_as_float(img)
    gray = transform.resize(gray, img_size, anti_aliasing=True)
    try:
        thresh = threshold_otsu(gray)
    except:
        thresh = 0.5
    binary = gray < thresh
    if binary.sum() < img_size[0] * img_size[1] * 0.02:
        binary = ~binary
    binary = binary_closing(binary, disk(3))
    binary = binary_opening(binary, disk(2))
    binary = remove_small_objects(binary.astype(bool), min_size=100)
    return binary.astype(np.uint8)

def extract_contour(binary, n_points=CONTOUR_POINTS):
    contours_list = find_contours(binary.astype(float), 0.5)
    if not contours_list:
        return np.zeros((n_points, 2))
    contour = max(contours_list, key=len)
    diffs = np.diff(contour, axis=0)
    arc = np.r_[0, np.cumsum(np.sqrt((diffs**2).sum(axis=1)))]
    if arc[-1] < 1e-8:
        return np.zeros((n_points, 2))
    u = np.linspace(0, arc[-1], n_points, endpoint=False)
    return np.column_stack([np.interp(u, arc, contour[:, 0]),
                            np.interp(u, arc, contour[:, 1])])

def center_and_scale(c):
    c = c - c.mean(axis=0)
    r = np.sqrt((c**2).sum(axis=1)).max()
    return c / r if r > 1e-8 else c

def compute_curvature(contour):
    x, y = contour[:, 1], contour[:, 0]
    x1 = np.gradient(x); y1 = np.gradient(y)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    return (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5

print('Loading MPEG-7 shapes...')
binaries, contours, curvatures, all_labels = [], [], [], []
for path, label in tqdm(samples, desc='Loading'):
    try:
        bimg = load_and_binarize(path)
        cnt = center_and_scale(extract_contour(bimg))
        kap = compute_curvature(cnt)
        binaries.append(bimg); contours.append(cnt)
        curvatures.append(kap); all_labels.append(label)
    except Exception as e:
        print(f'Failed: {path} — {e}')

le = LabelEncoder()
y = le.fit_transform(all_labels)
le.classes_ = np.array([str(c) for c in le.classes_])
n_classes = len(le.classes_)
print(f'Loaded: {len(contours)} | Classes: {n_classes}')

In [ ]:
# ── Figure 1: MPEG-7 Dataset Samples ────────────────────────────
n_display = min(20, n_classes)
fig, axes = plt.subplots(2, 10, figsize=(20, 4))
axes = axes.flatten()
fig.suptitle('Figure 1: MPEG-7 CE-Shape-1 Benchmark Dataset\n'
             f'{n_classes} Classes x 20 Instances = {len(contours)} Binary Silhouettes',
             fontsize=14, fontweight='bold', y=1.02)
for i in range(n_display):
    idx = np.where(y == i)[0][0]
    axes[i].imshow(binaries[idx], cmap='gray')
    axes[i].set_title(le.classes_[i].capitalize(), fontsize=8, fontweight='bold')
    axes[i].axis('off')
for ax in axes[n_display:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/fig1_mpeg7_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── Cell 5: Baseline Shape Descriptors ─────────────────────────────

def fourier_descriptor(contour, n_coeff=64):
    r = np.sqrt((contour**2).sum(axis=1))
    F = np.fft.fft(r); mag = np.abs(F)
    denom = mag[1] if mag[1] > 1e-8 else mag.max() + 1e-12
    mag_n = mag / denom
    return np.concatenate([mag_n[1:n_coeff + 1][:-1], np.angle(F)[1:17]])

def wavelet_descriptor(contour):
    r = np.sqrt((contour**2).sum(axis=1))
    r = r - r.mean()
    feats = []
    for wv_name in ['db4', 'haar', 'sym4']:
        safe_level = max(1, min(5, pywt.dwt_max_level(len(r), wv_name)))
        coeffs = pywt.wavedec(r, wv_name, level=safe_level, mode='periodization')
        energies = np.array([np.sum(c**2) for c in coeffs])
        feats.append(energies / (energies.sum() + 1e-12))
    max_len = max(len(f) for f in feats)
    return np.concatenate([np.pad(f, (0, max_len - len(f))) for f in feats])

def simple_hybrid_descriptor(contour):
    return np.concatenate([fourier_descriptor(contour, 64), wavelet_descriptor(contour)])

def zernike_moments(binary, max_order=10):
    h, w = binary.shape
    yg, xg = np.mgrid[-1:1:1j*h, -1:1:1j*w]
    r = np.sqrt(xg**2 + yg**2); theta = np.arctan2(yg, xg)
    mask = (r <= 1.) & (binary > 0); moments = []
    for n in range(max_order + 1):
        for m in range(-n, n + 1, 2):
            if (n - abs(m)) % 2 != 0: continue
            R = np.zeros_like(r)
            for s in range((n - abs(m)) // 2 + 1):
                c = ((-1)**s * scipy.special.factorial(n - s)) / (
                    scipy.special.factorial(s) *
                    scipy.special.factorial((n + abs(m)) // 2 - s) *
                    scipy.special.factorial((n - abs(m)) // 2 - s) + 1e-300)
                R += c * r**(n - 2*s)
            V = R * np.exp(-1j * m * theta)
            moments.append(np.abs(np.sum(V[mask] * binary[mask]) * (n + 1) / np.pi))
    return np.array(moments[:36])

def shape_context(contour, n_r=5, n_theta=12):
    N = len(contour); step = max(1, N // 64); pts = contour[::step]; n = len(pts)
    dx = pts[:, 1:2] - pts[np.newaxis, :, 1]; dy = pts[:, 0:1] - pts[np.newaxis, :, 0]
    dist = np.sqrt(dx**2 + dy**2 + 1e-12); angles = np.arctan2(dy, dx)
    log_dist = np.log(dist / (dist.max() + 1e-12) + 1e-12)
    r_bins = np.linspace(log_dist.min() - 0.01, 0.01, n_r + 1)
    t_bins = np.linspace(-np.pi, np.pi, n_theta + 1)
    H_g = np.zeros(n_r * n_theta)
    for i in range(n):
        mi = np.arange(n) != i
        H, _, _ = np.histogram2d(log_dist[i, mi], angles[i, mi], bins=[r_bins, t_bins])
        H_g += H.flatten()
    return H_g / (H_g.sum() + 1e-12)

def curvature_scale_space(contour, sigmas=None):
    if sigmas is None:
        sigmas = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
    x, yc = contour[:, 1], contour[:, 0]; feats = []
    for sigma in sigmas:
        xs = ndimage.gaussian_filter1d(x, sigma, mode='wrap')
        ys = ndimage.gaussian_filter1d(yc, sigma, mode='wrap')
        x1 = np.gradient(xs); x2 = np.gradient(x1)
        y1 = np.gradient(ys); y2 = np.gradient(y1)
        k = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
        feats += [float(np.sum(np.diff(np.sign(k)) != 0)), np.mean(np.abs(k))]
    return np.array(feats)

def hog_descriptor(binary):
    return feature.hog(binary.astype(np.float32), orientations=9,
                       pixels_per_cell=(16, 16), cells_per_block=(1, 1),
                       feature_vector=True)

print('All baseline descriptors defined.')
test_hog = hog_descriptor(binaries[0])
print(f'HOG dim: {len(test_hog)}')

In [ ]:
# ── Cell 6: AMST v11 Full Descriptor (466 dims) ────────────────────────
# C1 APCFW+  : 64 + 16 + 32 + 32 = 144
# C2 Topo MR : 3 resolutions x 30  = 90
# C3 SPD+    : 20x20 upper-tri     = 210
# C5 Compl.  : 22
# TOTAL      : 144 + 90 + 210 + 22 = 466

# ── C1: Adaptive Phase-Coherent Fourier-Wavelet Fusion (144-dim) ────────
# v11: Robust adaptive wavelet selection with consistency check
def phase_coherent_fourier_wavelet_plus(contour, K_F=64, n_wbands=32):
    r = np.sqrt((contour**2).sum(axis=1))
    F = np.fft.fft(r); mags = np.abs(F); phases = np.angle(F)
    denom = mags[1] if mags[1] > 1e-8 else mags.max() + 1e-12
    mag_n = mags / denom
    fd = mag_n[1:K_F + 1]; ph = phases[1:17]
    n_star = int(np.argmax(mags[1:K_F + 1])) + 1
    rho = n_star / K_F
    k = rho * K_F
    wv = 'db4' if rho < 0.10 else ('db2' if rho < 0.25 else 'haar')
    x, yc = contour[:, 1], contour[:, 0]
    x1 = np.gradient(x); y1 = np.gradient(yc)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
    kc = kappa - kappa.mean()
    max_lv = pywt.dwt_max_level(len(kc), wv)
    L = max(2, min(6, max_lv))
    coeffs = pywt.wavedec(kc, wv, level=L, mode='periodization')
    energies = np.array([np.sum(c**2) for c in coeffs])
    E = energies / (energies.sum() + 1e-12); n_actual = len(E)
    h_idx = np.array([min(l * max(1, int(k) // max(n_actual, 1)) + 1, K_F - 1) for l in range(n_actual)])
    cos_ph = np.abs(np.cos(phases[h_idx]))
    Omega = E * cos_ph + 1e-12; Omega /= Omega.sum()
    xi = np.linspace(0, 1, n_actual); xo = np.linspace(0, 1, n_wbands)
    Omega32 = interp1d(xi, Omega, kind='linear', fill_value='extrapolate')(xo)
    Omega32 = np.maximum(Omega32, 0); Omega32 /= Omega32.sum() + 1e-12
    angles = np.arctan2(contour[:, 0], contour[:, 1])
    ang_hist, _ = np.histogram(angles, bins=32, range=(-np.pi, np.pi))
    ang_dist = ang_hist / (ang_hist.sum() + 1e-12)
    return np.concatenate([fd, ph, Omega32, ang_dist])

# ── C2: Multi-Resolution Topology (90-dim) ────────────────────
# v11: Better delay embedding parameters, robust persistence vectorization
def multi_resolution_topology(contour):
    res_points = [128, 256, 512]
    taus = [3, 5, 8]
    all_feats = []
    for res_idx, n_pts in enumerate(res_points):
        if len(contour) > n_pts:
            idx = np.linspace(0, len(contour) - 1, n_pts, dtype=int)
            c_sub = contour[idx]
        else:
            c_sub = contour
        x, yc = c_sub[:, 1], c_sub[:, 0]
        x1 = np.gradient(x); y1 = np.gradient(yc)
        x2 = np.gradient(x1); y2 = np.gradient(y1)
        kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
        kn = (kappa - kappa.min()) / (kappa.max() - kappa.min() + 1e-12)
        N = len(kn)
        tau = min(taus[res_idx], max(2, N // 8))
        if N <= tau * 2 + 2:
            all_feats.append(np.zeros(30)); continue
        Xk = np.column_stack([kn[:N - tau], kn[tau:]])
        if len(Xk) > 200:
            idxs = np.linspace(0, len(Xk) - 1, 200, dtype=int)
            Xk = Xk[idxs]
        try:
            dgms = ripser_main(Xk, maxdim=1)['dgms']
        except:
            all_feats.append(np.zeros(30)); continue
        def vect(dgm, k=7):
            fin = dgm[dgm[:, 1] < np.inf]
            if len(fin) == 0:
                return np.zeros(k), np.zeros(k), np.zeros(6)
            lt = np.sort(fin[:, 1] - fin[:, 0])[::-1]
            bt = np.sort(fin[:, 0])
            lp = np.zeros(k); lp[:min(len(lt), k)] = lt[:k]
            bp = np.zeros(k); bp[:min(len(bt), k)] = bt[:k]
            tot = lt.sum(); mx = lt[0] if len(lt) > 0 else 0
            betti = float((lt > 0.01 * tot).sum()) if tot > 0 else 0
            ent = -np.sum(lt / tot * np.log(lt / tot + 1e-12)) if tot > 0 else 0
            med = float(np.median(lt)) if len(lt) > 0 else 0
            return lp, bp, np.array([tot, mx, betti, ent, med, float(len(fin))])
        lt0, bt0, st0 = vect(dgms[0])
        lt1, bt1, st1 = vect(dgms[1])
        feat = np.concatenate([lt0, lt1, bt0[:4], bt1[:4], st0, st1[:4]])
        out = np.zeros(30)
        out[:min(len(feat), 30)] = feat[:30]
        all_feats.append(out)
    return np.concatenate(all_feats)

# ── C3: SPD Manifold Features (210-dim) ──────────────────
def spd_manifold_features_v8(contour):
    r = np.sqrt((contour**2).sum(axis=1))
    x, yc = contour[:, 1], contour[:, 0]; N = len(r)
    x1 = np.gradient(x); y1 = np.gradient(yc)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
    t = np.linspace(0, 2 * np.pi, N, endpoint=False)
    rows = [
        r - r.mean(), x - x.mean(), yc - yc.mean(), kappa,
        np.cos(t), np.sin(t),
        ndimage.gaussian_filter1d(r - r.mean(), 2, mode='wrap'),
        ndimage.gaussian_filter1d(r - r.mean(), 8, mode='wrap'),
        ndimage.gaussian_filter1d(kappa, 2, mode='wrap'),
        ndimage.gaussian_filter1d(kappa, 4, mode='wrap'),
        ndimage.gaussian_filter1d(kappa, 8, mode='wrap'),
        np.gradient(kappa),
        ndimage.gaussian_filter1d(r - r.mean(), 4, mode='wrap'),
        ndimage.gaussian_filter1d(kappa, 1, mode='wrap'),
        np.abs(kappa), kappa**2,
        np.sqrt(np.abs(kappa) + 1e-12),
        np.sin(2 * t), np.cos(2 * t),
        np.arctan2(yc, x),
    ]
    d_spd = 20
    fm = np.array(rows[:d_spd], dtype=float)
    fm -= fm.mean(axis=1, keepdims=True)
    norms = np.linalg.norm(fm, axis=1, keepdims=True)
    fm /= (norms + 1e-12)
    S = (fm @ fm.T) / (N - 1) + 1e-4 * np.eye(d_spd)
    ev, evec = np.linalg.eigh(S)
    ev = np.maximum(ev, 1e-8)
    log_S = evec @ np.diag(np.log(ev)) @ evec.T
    return log_S[np.triu_indices(d_spd)]

# ── C5: Shape Complexity Features (22-dim) ───────────────
def shape_complexity_features(contour, binary):
    feats = []
    hull = ConvexHull(contour)
    hull_area = hull.volume; hull_perim = hull.area
    contour_area = np.abs(np.sum(contour[:-1, 0] * contour[1:, 1]
                                 - contour[1:, 0] * contour[:-1, 1])) / 2
    contour_perim = np.sum(np.sqrt(np.diff(contour[:, 0])**2 + np.diff(contour[:, 1])**2))
    feats.extend([hull_perim / (contour_perim + 1e-12),
                  contour_area / (hull_area + 1e-12),
                  4 * np.pi * contour_area / (contour_perim**2 + 1e-12)])
    labeled = measure.label(binary)
    props = regionprops(labeled)
    if props:
        p = props[0]
        feats.append(float(p.euler_number))
        feats.append(p.major_axis_length / (p.minor_axis_length + 1e-12))
        feats.append(p.extent); feats.append(p.eccentricity)
        feats.append(p.equivalent_diameter_area / max(binary.shape))
        feats.append(p.perimeter / (contour_perim + 1e-12))
        feats.append(p.area / (binary.shape[0] * binary.shape[1]))
    else:
        feats.extend([0.0] * 7)
    x, yc = contour[:, 1], contour[:, 0]
    x1 = np.gradient(x); y1 = np.gradient(yc)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
    feats.extend([np.mean(kappa), np.std(kappa), np.max(np.abs(kappa)),
                  np.sum(np.diff(np.sign(kappa)) != 0) / len(kappa),
                  float(scipy.stats.skew(kappa)),
                  float(scipy.stats.kurtosis(kappa)),
                  float(np.percentile(np.abs(kappa), 90))])
    d = np.sqrt((contour**2).sum(axis=1))
    feats.extend([np.std(d), float(np.max(d) - np.min(d)), np.mean(d),
                  float(np.percentile(d, 25)), float(np.percentile(d, 75))])
    return np.array(feats, dtype=float)

# ── C4: Multi-Head Fisher Band Attention (fitted per fold — no leakage) ──
class MultiHeadFisherBandAttention:
    def __init__(self, n_heads=8, n_bands=16, top_k_ratio=0.50):
        self.n_heads = n_heads; self.n_bands = n_bands
        self.top_k_ratio = top_k_ratio
        self.attention_weights = None; self.band_boundaries = None
    def fit(self, X_train, y_train):
        N, D = X_train.shape
        bs = D // self.n_bands
        self.band_boundaries = [(i * bs, min((i + 1) * bs, D)) for i in range(self.n_bands)]
        cls = np.unique(y_train); k = max(1, int(self.n_bands * self.top_k_ratio))
        head_weights = []
        for h in range(self.n_heads):
            np.random.seed(h * 17 + 3)
            sub = np.random.choice(N, int(0.8 * N), replace=False) if N > 20 else np.arange(N)
            Xs, ys = X_train[sub], y_train[sub]
            gms = Xs.mean(axis=0); scores = []
            for b0, b1 in self.band_boundaries:
                Xb = Xs[:, b0:b1]; gmb = gms[b0:b1]; SB = SW = 0.0
                for c in cls:
                    mk = ys == c
                    if mk.sum() < 2: continue
                    mc = Xb[mk].mean(axis=0)
                    SB += mk.sum() * np.dot(mc - gmb, mc - gmb)
                    SW += np.sum((Xb[mk] - mc)**2)
                scores.append(SB / (SW + 1e-8))
            sc = np.array(scores); sc = np.exp(sc - sc.max()); sc /= sc.sum()
            sp = np.zeros(self.n_bands)
            top = np.argsort(sc)[::-1][:k]; sp[top] = sc[top]; sp /= sp.sum() + 1e-12
            head_weights.append(sp)
        avg = np.mean(head_weights, axis=0); avg /= avg.sum() + 1e-12
        self.attention_weights = np.zeros(D)
        for i, (b0, b1) in enumerate(self.band_boundaries):
            self.attention_weights[b0:b1] = avg[i]
        return self
    def transform(self, X):
        if self.attention_weights is None:
            raise ValueError('Call fit() first')
        return X * self.attention_weights[np.newaxis, :]

# ── Full AMST v11 Descriptor ─────────────────────────
def amst_descriptor_v11(contour, binary):
    c1 = phase_coherent_fourier_wavelet_plus(contour)
    c2 = multi_resolution_topology(contour)
    c3 = spd_manifold_features_v8(contour)
    c5 = shape_complexity_features(contour, binary)
    return np.concatenate([c1, c2, c3, c5])

# Dimension verification
test_amst = amst_descriptor_v11(contours[0], binaries[0])
c1_d = len(phase_coherent_fourier_wavelet_plus(contours[0]))
c2_d = len(multi_resolution_topology(contours[0]))
c3_d = len(spd_manifold_features_v8(contours[0]))
c5_d = len(shape_complexity_features(contours[0], binaries[0]))
print(f'  C1 APCFW+ : {c1_d} (expected 144)')
print(f'  C2 Topo MR: {c2_d} (expected 90)')
print(f'  C3 SPD+   : {c3_d} (expected 210)')
print(f'  C5 Compl. : {c5_d} (expected 22)')
print(f'  TOTAL     : {len(test_amst)} (expected {AMST_DIM})')
assert len(test_amst) == AMST_DIM, f'AMST dim wrong: {len(test_amst)} != {AMST_DIM}'
assert c5_d == 22, f'C5 dim wrong: {c5_d}'
print(f'AMST v11 descriptor: {AMST_DIM}-dim — verified.')

In [ ]:
# ── Cell 7: Deep Learning Baseline (CNN) ────────────────────────────
# A simple CNN trained from scratch on the MPEG-7 binary silhouettes.
# Uses the same 5-fold stratified CV for fair comparison.

if USE_CNN:
    class SilhouetteDataset(Dataset):
        def __init__(self, images, labels, transform=None):
            self.images = images
            self.labels = labels
            self.transform = transform
        def __len__(self):
            return len(self.images)
        def __getitem__(self, idx):
            img = self.images[idx].astype(np.float32).reshape(1, 128, 128)
            if self.transform:
                img = self.transform(img)
            return torch.from_numpy(img), self.labels[idx]

    class SimpleCNN(nn.Module):
        def __init__(self, num_classes):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            )
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128 * 16 * 16, 256), nn.ReLU(), nn.Dropout(0.5),
                nn.Linear(256, num_classes)
            )
        def forward(self, x):
            x = self.features(x)
            return self.classifier(x)

    def extract_cnn_features(model, loader, device):
        model.eval()
        feats, labs = [], []
        with torch.no_grad():
            for x, y in loader:
                x = x.to(device)
                feat = model.features(x)
                feat = feat.view(feat.size(0), -1).cpu().numpy()
                feats.append(feat)
                labs.append(y.numpy())
        return np.concatenate(feats), np.concatenate(labs)

    def train_cnn_fold(train_idx, val_idx, X_raw, y, num_classes, device, epochs=30):
        X_train_raw = np.array([X_raw[i] for i in train_idx])
        y_train = y[train_idx]
        X_val_raw = np.array([X_raw[i] for i in val_idx])
        y_val = y[val_idx]

        train_ds = SilhouetteDataset(X_train_raw, y_train)
        val_ds = SilhouetteDataset(X_val_raw, y_val)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=32)

        model = SimpleCNN(num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

        best_acc = 0.0
        patience_counter = 0
        for epoch in range(epochs):
            model.train()
            for x, lbl in train_loader:
                x, lbl = x.to(device), lbl.to(device)
                optimizer.zero_grad()
                loss = criterion(model(x), lbl)
                loss.backward()
                optimizer.step()
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for x, lbl in val_loader:
                    x, lbl = x.to(device), lbl.to(device)
                    out = model(x)
                    _, pred = out.max(1)
                    total += lbl.size(0)
                    correct += (pred == lbl).sum().item()
            acc = correct / total
            scheduler.step(1.0 - acc)
            if acc > best_acc:
                best_acc = acc
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= 10:
                break

        # Extract features from penultimate layer
        train_feats, train_labels = extract_cnn_features(model, train_loader, device)
        val_feats, val_labels = extract_cnn_features(model, val_loader, device)
        return train_feats, val_feats, best_acc

    print(f'PyTorch CNN baseline ready (GPU: {torch.cuda.is_available()})')
    print('CNN features will be extracted per fold during CV evaluation.')
else:
    print('CNN baseline not available (PyTorch not found or no GPU).')
    print('Falling back to MLP classifier on HOG features as additional baseline.')

In [ ]:
# ── Cell 8: Feature Extraction (All Methods) ───────────────────────────
print(f'Extracting features from {len(contours)} images...')
N = len(contours)
fd_f=[]; wd_f=[]; hy_f=[]; ze_f=[]; cs_f=[]; sc_f=[]; hg_f=[]; am_f=[]
for i in tqdm(range(N), desc='Features'):
    cnt = contours[i]; bimg = binaries[i]
    try:    fd_f.append(fourier_descriptor(cnt))
    except: fd_f.append(np.zeros(79))
    try:    wd_f.append(wavelet_descriptor(cnt))
    except: wd_f.append(np.zeros(18))
    try:    hy_f.append(simple_hybrid_descriptor(cnt))
    except: hy_f.append(np.zeros(97))
    try:    ze_f.append(zernike_moments(bimg))
    except: ze_f.append(np.zeros(36))
    try:    cs_f.append(curvature_scale_space(cnt))
    except: cs_f.append(np.zeros(20))
    try:    sc_f.append(shape_context(cnt))
    except: sc_f.append(np.zeros(60))
    try:    hg_f.append(hog_descriptor(bimg))
    except: hg_f.append(np.zeros(576))
    try:    am_f.append(amst_descriptor_v11(cnt, bimg))
    except: am_f.append(np.zeros(AMST_DIM))

X_fd      = np.array(fd_f)
X_wd      = np.array(wd_f)
X_hybrid  = np.array(hy_f)
X_zernike = np.array(ze_f)
X_css     = np.array(cs_f)
X_sc      = np.array(sc_f)
X_hog     = np.array(hg_f)
X_amst    = np.nan_to_num(np.array(am_f))

print('\nFeature dimensions:')
for nm, X in [('Fourier', X_fd), ('Wavelet', X_wd), ('F+W Hybrid', X_hybrid),
              ('Zernike', X_zernike), ('CSS', X_css), ('Shape Context', X_sc),
              ('HOG', X_hog), (f'AMST v11 ({AMST_DIM}d)', X_amst)]:
    print(f'  {nm:25s}: {X.shape}')
assert X_amst.shape[1] == AMST_DIM, f'AMST dim: {X_amst.shape[1]} != {AMST_DIM}'
print('All features extracted.')

In [ ]:
# ── Cell 9: Evaluation Functions ────────────────────────────────
N_SPLITS = 5

def find_best_svm(X_tr, y_tr):
    param_grid = {'C': [0.1, 1, 10, 100, 500, 1000],
                  'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]}
    gs = GridSearchCV(SVC(kernel='rbf', decision_function_shape='ovr'),
                      param_grid, cv=min(3, len(np.unique(y_tr))),
                      scoring='accuracy', n_jobs=-1)
    gs.fit(X_tr, y_tr)
    return gs.best_params_

def evaluate_descriptor_svm(X, y, name, n_splits=N_SPLITS, use_attention=False):
    X = np.nan_to_num(X.copy())
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    accs=[]; f1s=[]; precs=[]; recs=[]; fold_accs=[]
    for tr, te in skf.split(X, y):
        X_tr, X_te = X[tr], X[te]; y_tr, y_te = y[tr], y[te]
        if use_attention:
            attn = MultiHeadFisherBandAttention(8, 16, 0.50)
            attn.fit(X_tr, y_tr)
            X_tr = attn.transform(X_tr); X_te = attn.transform(X_te)
        scaler = RobustScaler()
        X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)
        bp = find_best_svm(X_tr_s, y_tr)
        clf = SVC(kernel='rbf', decision_function_shape='ovr', **bp)
        clf.fit(X_tr_s, y_tr); yp = clf.predict(X_te_s)
        accs.append(accuracy_score(y_te, yp))
        f1s.append(f1_score(y_te, yp, average='macro', zero_division=0))
        precs.append(precision_score(y_te, yp, average='macro', zero_division=0))
        recs.append(recall_score(y_te, yp, average='macro', zero_division=0))
        fold_accs.append(accuracy_score(y_te, yp))
    return {'Method': name, 'Accuracy': np.mean(accs), 'Accuracy_std': np.std(accs),
            'F1_macro': np.mean(f1s), 'Precision_macro': np.mean(precs),
            'Recall_macro': np.mean(recs), 'Dim': X.shape[1], 'fold_accs': fold_accs}

# ── Fixed stacking: proper calibration with probability clipping ──
def stacking_ensemble_predict(X_train, y_train, X_test):
    rf = RandomForestClassifier(n_estimators=300, max_depth=12,
                                random_state=SEED, n_jobs=-1)
    xgb_clf = (xgb.XGBClassifier(n_estimators=300, max_depth=6,
               learning_rate=0.1, random_state=SEED, n_jobs=-1, verbosity=0,
               use_label_encoder=False, eval_metric='mlogloss')
               if XGB_AVAILABLE else None)
    rf_preds = cross_val_predict(rf, X_train, y_train, cv=5, method='predict_proba')
    if xgb_clf is not None:
        xgb_preds = cross_val_predict(xgb_clf, X_train, y_train, cv=5, method='predict_proba')
        meta_train = np.column_stack([rf_preds, xgb_preds])
    else:
        meta_train = rf_preds
    rf.fit(X_train, y_train); rf_test = rf.predict_proba(X_test)
    if xgb_clf is not None:
        xgb_clf.fit(X_train, y_train)
        meta_test = np.column_stack([rf_test, xgb_clf.predict_proba(X_test)])
    else:
        meta_test = rf_test
    meta = LogisticRegression(multi_class='multinomial', max_iter=2000,
                              C=0.5, random_state=SEED)
    meta.fit(meta_train, y_train)
    return meta.predict(meta_test)

def evaluate_descriptor_stacking(X, y, name, n_splits=N_SPLITS, use_attention=False):
    X = np.nan_to_num(X.copy())
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    accs=[]; f1s=[]; fold_accs=[]
    for tr, te in skf.split(X, y):
        X_tr, X_te = X[tr], X[te]; y_tr, y_te = y[tr], y[te]
        if use_attention:
            attn = MultiHeadFisherBandAttention(8, 16, 0.50)
            attn.fit(X_tr, y_tr)
            X_tr = attn.transform(X_tr); X_te = attn.transform(X_te)
        scaler = RobustScaler()
        X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)
        yp = stacking_ensemble_predict(X_tr_s, y_tr, X_te_s)
        accs.append(accuracy_score(y_te, yp))
        f1s.append(f1_score(y_te, yp, average='macro', zero_division=0))
        fold_accs.append(accuracy_score(y_te, yp))
    return {'Method': name, 'Accuracy': np.mean(accs), 'Accuracy_std': np.std(accs),
            'F1_macro': np.mean(f1s), 'Dim': X.shape[1], 'fold_accs': fold_accs}

print('Evaluation functions ready.')

In [ ]:
# ── Cell 10: 5-Fold CV Evaluation (Fair Comparison) ──────────────────
print('Running 5-fold CV (ALL methods SVM-RBF, attention per-fold)...\n')
svm_results = []
method_list = [
    (X_fd,      'Fourier Descriptor',  False),
    (X_wd,      'Wavelet Descriptor',  False),
    (X_hybrid,  'Simple F+W Hybrid',   False),
    (X_zernike, 'Zernike Moments',     False),
    (X_css,     'CSS Descriptor',      False),
    (X_sc,      'Shape Context',       False),
    (X_hog,     'HOG',                 False),
    (X_amst,    'AMST (Proposed)',      True),
]
for X_feat, nm, use_attn in method_list:
    res = evaluate_descriptor_svm(X_feat, y, nm, use_attention=use_attn)
    svm_results.append(res)
    print(f'{nm:28s} | Acc: {res["Accuracy"]:.4f}±{res["Accuracy_std"]:.4f} | F1: {res["F1_macro"]:.4f}')

svm_df = pd.DataFrame(svm_results)
amst_acc_svm  = svm_df[svm_df['Method'] == 'AMST (Proposed)']['Accuracy'].values[0] * 100
best_base_svm = svm_df[svm_df['Method'] != 'AMST (Proposed)']['Accuracy'].max() * 100

print(f'\n=== FAIR Comparison (SVM-RBF, same classifier for all) ===')
print(f'AMST (SVM)   : {amst_acc_svm:.2f}%')
print(f'Best Baseline: {best_base_svm:.2f}%')
print(f'AMST Gain    : +{amst_acc_svm - best_base_svm:.2f} pp')

# Stacking ensemble (fixed)
print('\nRunning AMST stacking ensemble...')
stack_result = evaluate_descriptor_stacking(X_amst, y, 'AMST (Stacking)', use_attention=True)
amst_acc_stack = stack_result['Accuracy'] * 100
stack_gain = amst_acc_stack - amst_acc_svm
print(f'AMST (Stacking): {amst_acc_stack:.2f}% | Stacking gain: +{stack_gain:.2f} pp')

all_results_df = pd.DataFrame(svm_results + [stack_result])
print('\nAll results:')
print(svm_df[['Method','Accuracy','Accuracy_std','F1_macro','Dim']].to_string(index=False))

In [ ]:
# ── Cell 11: Statistical Significance (Bonferroni-corrected) ───────────
amst_folds = np.array(svm_df[svm_df['Method'] == 'AMST (Proposed)']['fold_accs'].values[0])
N_COMPARISONS = len(svm_df) - 1  # excluding AMST itself
ALPHA = 0.05
BONF_ALPHA = ALPHA / N_COMPARISONS

print(f'Paired t-test: AMST (SVM) vs each baseline')
print(f'Bonferroni correction: {N_COMPARISONS} comparisons, alpha={BONF_ALPHA:.5f}\n')
print(f'{"Method":<28} {"AMST%":>9} {"Base%":>9} {"Delta(pp)":>10} {"p-val":>10} {"Bonf.sig?":>10}')
print('-' * 76)
stat_rows = []
for _, row in svm_df.iterrows():
    nm = row['Method']
    if nm == 'AMST (Proposed)': continue
    bf = np.array(row['fold_accs'])
    t_stat, p_val = scipy.stats.ttest_rel(amst_folds, bf)
    delta = (amst_folds.mean() - bf.mean()) * 100
    sig_raw = p_val < ALPHA
    sig_bonf = p_val < BONF_ALPHA
    sig_str = '*' if sig_bonf else ('o' if sig_raw else ' ')
    print(f'{nm:<28} {amst_folds.mean()*100:>8.2f}% {bf.mean()*100:>8.2f}% {delta:>+9.2f}pp {p_val:>10.5f} {sig_str:>10}')
    stat_rows.append({'Baseline': nm, 'AMST_%': amst_folds.mean()*100,
                      'Base_%': bf.mean()*100, 'Delta_pp': delta,
                      'p_value': p_val, 'Significant_raw': sig_raw,
                      'Significant_bonf': sig_bonf})

stat_df = pd.DataFrame(stat_rows)
n_sig_bonf = stat_df['Significant_bonf'].sum()
print(f'\nAMST fold accs: {[f"{a:.3f}" for a in amst_folds]}')
print(f'AMST significant vs {n_sig_bonf}/{N_COMPARISONS} baselines (Bonferroni, p<{BONF_ALPHA:.5f})')

In [ ]:
# ── Cell 12: Ablation Study ─────────────────────────────────
def extract_ablation_variant(contour, binary, variant):
    c1 = phase_coherent_fourier_wavelet_plus(contour)
    c2 = multi_resolution_topology(contour)
    c3 = spd_manifold_features_v8(contour)
    c5 = shape_complexity_features(contour, binary)
    if variant == 'c1_only':    return c1
    elif variant == 'c1_c2':    return np.concatenate([c1, c2])
    elif variant == 'c1_c2_c3': return np.concatenate([c1, c2, c3])
    else:                        return np.concatenate([c1, c2, c3, c5])

print('Extracting ablation variants...')
variant_map = [
    ('c1_only',   'C1: APCFW+ only (144d)',          144),
    ('c1_c2',     'C1+C2: +Topology MR (234d)',       234),
    ('c1_c2_c3',  'C1+C2+C3: +SPD Manifold (444d)',  444),
    ('full_amst', 'Full AMST C1+C2+C3+C5 (466d)',    466),
]
abl_feats = {}
for vname, vlab, expected_dim in variant_map:
    feats = []
    for i in tqdm(range(len(contours)), desc=vlab, leave=False):
        try:    feats.append(extract_ablation_variant(contours[i], binaries[i], vname))
        except: feats.append(np.zeros(expected_dim))
    abl_feats[vlab] = np.nan_to_num(np.array(feats))

print('\nAblation 5-fold CV (SVM-RBF for all):')
abl_svm_results = []
for vlab, Xa in abl_feats.items():
    res = evaluate_descriptor_svm(Xa, y, vlab, use_attention=True)
    abl_svm_results.append(res)
    print(f'  {vlab:<45} Acc: {res["Accuracy"]*100:.2f}±{res["Accuracy_std"]*100:.2f}%')

abl_stack_res = evaluate_descriptor_stacking(abl_feats['Full AMST C1+C2+C3+C5 (466d)'],
                                              y, 'Full AMST + Stacking', use_attention=True)
abl_svm_results.append(abl_stack_res)
print(f'  {"Full AMST + Stacking":<45} Acc: {abl_stack_res["Accuracy"]*100:.2f}±{abl_stack_res["Accuracy_std"]*100:.2f}%')

abl_df = pd.DataFrame(abl_svm_results)

print('\nIncremental gains:')
abl_folds = [np.array(r['fold_accs']) for r in abl_svm_results]
abl_names = [r['Method'] for r in abl_svm_results]
for i in range(1, len(abl_folds) - 1):
    t_stat, p_val = scipy.stats.ttest_rel(abl_folds[i], abl_folds[i - 1])
    delta = (abl_folds[i].mean() - abl_folds[i - 1].mean()) * 100
    sig = '*' if p_val < ALPHA else ' '
    print(f'  {abl_names[i-1][:35]:<35} -> {abl_names[i][:35]:<35} {delta:+>7.2f}pp  p={p_val:.4f} {sig}')

In [ ]:
# ── Figure 2: Ablation Study ──────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
names = [r['Method'] for r in abl_svm_results]
accs = [r['Accuracy'] * 100 for r in abl_svm_results]
stds = [r['Accuracy_std'] * 100 for r in abl_svm_results]
colors = ['#AED6F1', '#5DADE2', '#2471A3', '#1A5276', '#E84040']
bars = ax.barh(names, accs, xerr=stds, color=colors, edgecolor='white', capsize=4, height=0.6)
for bar, acc in zip(bars, accs):
    ax.text(acc + 0.5, bar.get_y() + bar.get_height()/2, f'{acc:.2f}%', va='center', fontsize=9)
ax.set_xlabel('Accuracy (%)'); ax.set_title('Ablation Study — Incremental Component Gains')
ax.set_xlim(0, 105); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('/content/fig2_ablation.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 2 saved.')

In [ ]:
# ── Figure 3: Classification Performance ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

methods = svm_df['Method'].values
accs = svm_df['Accuracy'].values * 100
stds = svm_df['Accuracy_std'].values * 100
f1s = svm_df['F1_macro'].values * 100
colors = ['#5B7FA6'] * (len(methods) - 1) + ['#E84040']

# Accuracy
bars = axes[0].barh(methods, accs, xerr=stds, color=colors, edgecolor='white', capsize=4, height=0.65)
axes[0].set_xlabel('Accuracy (%)'); axes[0].set_xlim(0, 110)
axes[0].set_title('(A) Accuracy ± SD [Fair SVM-RBF comparison]')
axes[0].grid(axis='x', alpha=0.3)
for i, (bar, acc, std) in enumerate(zip(bars, accs, stds)):
    fw = 'bold' if i == len(methods) - 1 else 'normal'
    axes[0].text(acc + std + 0.5, bar.get_y() + bar.get_height()/2, f'{acc:.1f}%', va='center', fontsize=8, fontweight=fw)

# F1
bars2 = axes[1].barh(methods, f1s, color=colors, edgecolor='white', height=0.65)
axes[1].set_xlabel('Macro F1 (%)'); axes[1].set_xlim(0, 110)
axes[1].set_title('(B) Macro F1')
axes[1].grid(axis='x', alpha=0.3)
for i, (bar, f1) in enumerate(zip(bars2, f1s)):
    fw = 'bold' if i == len(methods) - 1 else 'normal'
    axes[1].text(f1 + 0.5, bar.get_y() + bar.get_height()/2, f'{f1:.1f}%', va='center', fontsize=8, fontweight=fw)

plt.tight_layout()
plt.savefig('/content/fig3_classification.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 3 saved.')

In [ ]:
# ── Figure 4: Confusion Matrix (AMST SVM) ───────────────────────
from sklearn.model_selection import StratifiedKFold
skf_cm = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
X_ac = np.nan_to_num(X_amst.copy()); all_yt=[]; all_yp=[]
for tr, te in skf_cm.split(X_ac, y):
    X_tr, X_te = X_ac[tr], X_ac[te]; y_tr, y_te = y[tr], y[te]
    attn_cm = MultiHeadFisherBandAttention(8, 16, 0.50)
    attn_cm.fit(X_tr, y_tr)
    X_tr = attn_cm.transform(X_tr); X_te = attn_cm.transform(X_te)
    sc_cm = RobustScaler()
    X_tr_s = sc_cm.fit_transform(X_tr); X_te_s = sc_cm.transform(X_te)
    bp_cm = find_best_svm(X_tr_s, y_tr)
    clf_cm = SVC(kernel='rbf', decision_function_shape='ovr', **bp_cm)
    clf_cm.fit(X_tr_s, y_tr)
    yp_cm = clf_cm.predict(X_te_s)
    all_yt.extend(y_te.tolist()); all_yp.extend(yp_cm.tolist())

cm_acc = accuracy_score(all_yt, all_yp)
cm = confusion_matrix(all_yt, all_yp)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
print(f'5-fold aggregated accuracy: {cm_acc*100:.2f}%')

# Show top-20 most confused classes
diag = np.diag(cm_pct)
worst_idx = np.argsort(diag)[:10]
print('\n10 worst-recognized classes:')
for idx in worst_idx:
    print(f'  {le.classes_[idx].capitalize():20s}: {diag[idx]:.1f}% recognition rate')

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label='Recognition Rate (%)')
cn = [le.classes_[i].capitalize() for i in range(70)]
ax.set_xticks(range(0, 70, 5)); ax.set_yticks(range(0, 70, 5))
ax.set_xticklabels(cn[::5], rotation=45, ha='right', fontsize=6)
ax.set_yticklabels(cn[::5], fontsize=6)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'AMST v11 Confusion Matrix (5-fold CV) | Acc: {cm_acc*100:.2f}%')
plt.tight_layout()
plt.savefig('/content/fig4_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 4 saved.')

In [ ]:
# ── Cell 13: Shape Retrieval (Bullseye Rating) ──────────────────────
def bullseye_rating(X, y, top_k=40):
    """Standard MPEG-7 Bullseye Rating: each image as query, count correct in top 40."""
    Xs = StandardScaler().fit_transform(np.nan_to_num(X))
    N = len(y)
    total_correct = 0
    total_possible = 0
    for qi in range(N):
        dists = np.sqrt(((Xs - Xs[qi])**2).sum(axis=1))
        ranked = np.argsort(dists)
        ranked = ranked[ranked != qi]
        matches = (y[ranked[:top_k]] == y[qi]).sum()
        n_same_class = (y == y[qi]).sum() - 1
        total_correct += matches
        total_possible += min(top_k, n_same_class)
    return total_correct / total_possible * 100

def compute_map(X, y):
    """Mean Average Precision for retrieval."""
    Xs = StandardScaler().fit_transform(np.nan_to_num(X))
    N = len(y); APs = []
    for qi in range(N):
        dists = np.sqrt(((Xs - Xs[qi])**2).sum(axis=1))
        ranked = np.argsort(dists)
        ranked = ranked[ranked != qi]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum() == 0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked) + 1)
        APs.append((cs / pos * rel).sum() / rel.sum())
    return np.mean(APs)

print('Computing retrieval metrics (leave-one-out)...')
ret_methods = [
    (X_fd,      'Fourier Descriptor'),
    (X_wd,      'Wavelet Descriptor'),
    (X_hybrid,  'Simple F+W Hybrid'),
    (X_zernike, 'Zernike Moments'),
    (X_sc,      'Shape Context'),
    (X_amst,    'AMST (Proposed)'),
]
ret_results = []
for Xf, nm in tqdm(ret_methods, desc='Retrieval'):
    bullseye = bullseye_rating(Xf, y, top_k=40)
    map_score = compute_map(Xf, y)
    ret_results.append({'Method': nm, 'Bullseye': bullseye, 'MAP': map_score})
    print(f'  {nm:28s} Bullseye: {bullseye:.2f}%  MAP: {map_score:.4f}')

ret_df = pd.DataFrame(ret_results)
best_bullseye = ret_df['Bullseye'].max()
amst_bullseye = ret_df[ret_df['Method'] == 'AMST (Proposed)']['Bullseye'].values[0]
print(f'\nAMST Bullseye: {amst_bullseye:.2f}% | Best: {best_bullseye:.2f}% | AMST is best: {amst_bullseye >= best_bullseye}')
print(ret_df.to_string(index=False))

In [ ]:
# ── Figure 5: Retrieval Precision-Recall Curves ───────────────────
def retrieval_pr(X, y):
    Xs = StandardScaler().fit_transform(np.nan_to_num(X))
    N = len(y); APs=[]; allP=[]; allR=[]
    for qi in range(N):
        dists = np.sqrt(((Xs - Xs[qi])**2).sum(axis=1))
        ranked = np.argsort(dists)[1:]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum() == 0: continue
        cs = np.cumsum(rel); pos = np.arange(1, len(ranked) + 1)
        APs.append((cs / pos * rel).sum() / rel.sum())
        allP.append(cs / pos); allR.append(cs / rel.sum())
    rg = np.linspace(0, 1, 20)
    ip = [np.interp(rg, r, p) for p, r in zip(allP, allR)]
    return rg, np.mean(ip, axis=0), np.mean(APs)

pr_curves = {}
cpr = ['#5B7FA6', '#E8A020', '#27AE60', '#8E44AD', '#34495E', '#E84040']
lpr = [1.5] * 5 + [2.8]
pr_methods = [
    (X_fd, 'Fourier'), (X_wd, 'Wavelet'), (X_hybrid, 'Hybrid'),
    (X_zernike, 'Zernike'), (X_sc, 'Shape Ctx'), (X_amst, 'AMST'),
]
for (Xf, nm), col in zip(pr_methods, cpr):
    rg, mp, MAP = retrieval_pr(Xf, y)
    pr_curves[nm] = (rg, mp, MAP)

fig, ax = plt.subplots(figsize=(10, 7))
for (nm, (rc, pr, MAP)), col, lw in zip(pr_curves.items(), cpr, lpr):
    ls = '-' if 'AMST' in nm else '--'
    ax.plot(rc, pr, color=col, lw=lw, label=f'{nm} (MAP={MAP:.3f})', ls=ls)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Shape Retrieval PR Curves (leave-one-out)')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('/content/fig5_retrieval_pr.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 5 saved.')

In [ ]:
# ── Cell 14: Robustness — Gaussian Contour Noise ─────────────────
noise_levels = [0.0, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25]
noise_methods = [
    (X_fd, 'Fourier'), (X_wd, 'Wavelet'), (X_hybrid, 'Hybrid'),
    (X_css, 'CSS'), (X_sc, 'Shape Ctx'), (X_amst, 'AMST'),
]
styles = [('--','o','#5B7FA6'), ('--','s','#E8A020'), ('--','^','#27AE60'),
          ('--','D','#8E44AD'), ('--','v','#34495E'), ('-','*','#E84040')]

def noisy_cnt(cnt, sigma):
    return center_and_scale(cnt + np.random.normal(0, sigma, cnt.shape))

def eval_noisy(X_orig, y, sigma, nm, n_runs=2):
    accs = []; D = X_orig.shape[1]
    for run in range(n_runs):
        np.random.seed(run * 7 + 13)
        itr, ite = np.split(np.random.permutation(len(y)), [int(0.7 * len(y))])
        Xte = []
        for i in ite:
            cn = noisy_cnt(contours[i], sigma)
            try:
                if 'AMST' in nm:    feat = amst_descriptor_v11(cn, binaries[i])
                elif 'Fourier' in nm: feat = fourier_descriptor(cn)
                elif 'Wavelet' in nm: feat = wavelet_descriptor(cn)
                elif 'Hybrid' in nm: feat = simple_hybrid_descriptor(cn)
                elif 'CSS' in nm:   feat = curvature_scale_space(cn)
                else:               feat = shape_context(cn)
                if len(feat) < D: feat = np.pad(feat, (0, D - len(feat)))
                elif len(feat) > D: feat = feat[:D]
            except: feat = np.zeros(D)
            Xte.append(feat)
        Xte = np.nan_to_num(np.array(Xte))
        sc_n = RobustScaler()
        clf_n = SVC(kernel='rbf', C=100, gamma='scale')
        clf_n.fit(sc_n.fit_transform(np.nan_to_num(X_orig[itr])), y[itr])
        accs.append(accuracy_score(y[ite], clf_n.predict(sc_n.transform(Xte))))
    return np.mean(accs)

print('Computing noise robustness...')
noise_res = {nm: [] for _, nm in noise_methods}
for sigma in tqdm(noise_levels[:4], desc='Noise (subset)'):
    for Xf, nm in noise_methods:
        noise_res[nm].append(eval_noisy(Xf, y, sigma, nm))
print('Done.')

In [ ]:
# ── Figure 6: Noise Robustness ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for (_, nm), (ls, mk, col) in zip(noise_methods, styles):
    lw = 2.8 if 'AMST' in nm else 1.5; ms = 10 if 'AMST' in nm else 7
    ax.plot(noise_levels[:4], [v * 100 for v in noise_res[nm]],
            ls=ls, marker=mk, color=col, lw=lw, ms=ms, label=nm)
ax.set_xlabel('Gaussian Noise \u03c3'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Robustness to Contour Noise')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(0, 105)
plt.tight_layout(); plt.savefig('/content/fig6_noise.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 6 saved.')

In [ ]:
# ── Cell 15: Robustness — Occlusion ──────────────────────────
occ_levels = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]

def occ_binary_fn(binary, frac):
    if frac == 0: return binary
    h, w = binary.shape; side = np.sqrt(frac)
    ch, cw = int(h * side), int(w * side)
    r0 = max(0, (h - ch) // 2); c0 = max(0, (w - cw) // 2)
    occ = binary.copy(); occ[r0:r0 + ch, c0:c0 + cw] = 0
    return occ

def eval_occ(X_orig, y, frac, nm, n_runs=2):
    accs = []; D = X_orig.shape[1]
    for run in range(n_runs):
        np.random.seed(run * 11 + 7)
        itr, ite = np.split(np.random.permutation(len(y)), [int(0.7 * len(y))])
        Xte = []
        for i in ite:
            ob = occ_binary_fn(binaries[i], frac)
            try:
                oc = center_and_scale(extract_contour(ob))
                if 'AMST' in nm:    feat = amst_descriptor_v11(oc, ob)
                elif 'Fourier' in nm: feat = fourier_descriptor(oc)
                elif 'Wavelet' in nm: feat = wavelet_descriptor(oc)
                elif 'Hybrid' in nm: feat = simple_hybrid_descriptor(oc)
                elif 'CSS' in nm:   feat = curvature_scale_space(oc)
                else:               feat = shape_context(oc)
                if len(feat) < D: feat = np.pad(feat, (0, D - len(feat)))
                elif len(feat) > D: feat = feat[:D]
            except: feat = np.zeros(D)
            Xte.append(feat)
        Xte = np.nan_to_num(np.array(Xte))
        sc_o = RobustScaler()
        clf_o = SVC(kernel='rbf', C=100, gamma='scale')
        clf_o.fit(sc_o.fit_transform(np.nan_to_num(X_orig[itr])), y[itr])
        accs.append(accuracy_score(y[ite], clf_o.predict(sc_o.transform(Xte))))
    return np.mean(accs)

print('Computing occlusion robustness...')
occ_res = {nm: [] for _, nm in noise_methods}
for frac in tqdm(occ_levels[:4], desc='Occlusion (subset)'):
    for Xf, nm in noise_methods:
        occ_res[nm].append(eval_occ(Xf, y, frac, nm))
print('Done.')

In [ ]:
# ── Figure 7: Occlusion Robustness ───────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for (_, nm), (ls, mk, col) in zip(noise_methods, styles):
    lw = 2.8 if 'AMST' in nm else 1.5; ms = 10 if 'AMST' in nm else 7
    ax.plot([f * 100 for f in occ_levels[:4]], [v * 100 for v in occ_res[nm]],
            ls=ls, marker=mk, color=col, lw=lw, ms=ms, label=nm)
ax.set_xlabel('Occlusion (% area)'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Robustness to Occlusion')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(0, 105)
plt.tight_layout(); plt.savefig('/content/fig7_occlusion.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 7 saved.')

In [ ]:
# ── Cell 16: Robustness — Rotation ────────────────────────
def rotate_contour(cnt, angle_deg):
    angle = np.radians(angle_deg)
    R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    return center_and_scale(cnt @ R.T)

def eval_rotation(X_orig, y, angle, nm, n_runs=2):
    accs = []; D = X_orig.shape[1]
    for run in range(n_runs):
        np.random.seed(run * 13 + 5)
        itr, ite = np.split(np.random.permutation(len(y)), [int(0.7 * len(y))])
        Xte = []
        for i in ite:
            rc = rotate_contour(contours[i], angle)
            try:
                if 'AMST' in nm:    feat = amst_descriptor_v11(rc, binaries[i])
                elif 'Fourier' in nm: feat = fourier_descriptor(rc)
                elif 'Wavelet' in nm: feat = wavelet_descriptor(rc)
                elif 'Hybrid' in nm: feat = simple_hybrid_descriptor(rc)
                elif 'CSS' in nm:   feat = curvature_scale_space(rc)
                else:               feat = shape_context(rc)
                if len(feat) < D: feat = np.pad(feat, (0, D - len(feat)))
                elif len(feat) > D: feat = feat[:D]
            except: feat = np.zeros(D)
            Xte.append(feat)
        Xte = np.nan_to_num(np.array(Xte))
        sc_r = RobustScaler()
        clf_r = SVC(kernel='rbf', C=100, gamma='scale')
        clf_r.fit(sc_r.fit_transform(np.nan_to_num(X_orig[itr])), y[itr])
        accs.append(accuracy_score(y[ite], clf_r.predict(sc_r.transform(Xte))))
    return np.mean(accs)

rot_methods = [
    (X_fd, 'Fourier'), (X_wd, 'Wavelet'), (X_hybrid, 'Hybrid'),
    (X_sc, 'Shape Ctx'), (X_amst, 'AMST'),
]
rot_styles = [('--','o','#5B7FA6'), ('--','s','#E8A020'), ('--','^','#27AE60'),
              ('--','v','#34495E'), ('-','*','#E84040')]
angles = [0, 15, 30, 45, 60, 90]
print('Computing rotation robustness...')
rot_res = {nm: [] for _, nm in rot_methods}
for angle in tqdm(angles[:4], desc='Rotation (subset)'):
    for Xf, nm in rot_methods:
        rot_res[nm].append(eval_rotation(Xf, y, angle, nm))
print('Done.')

In [ ]:
# ── Figure 8: Rotation Robustness ────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for (_, nm), (ls, mk, col) in zip(rot_methods, rot_styles):
    lw = 2.8 if 'AMST' in nm else 1.5; ms = 10 if 'AMST' in nm else 7
    ax.plot(angles[:4], [v * 100 for v in rot_res[nm]],
            ls=ls, marker=mk, color=col, lw=lw, ms=ms, label=nm)
ax.set_xlabel('Rotation (degrees)'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Robustness to Rotation')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(0, 105)
plt.tight_layout(); plt.savefig('/content/fig8_rotation.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 8 saved.')

In [ ]:
# ── Figure 9: Dashboard ────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f'AMST v11 Performance Dashboard — MPEG-7 CE-Shape-1\n'
             f'({AMST_DIM}-dim | Fair SVM comparison | Bonferroni-corrected | 5-fold CV)',
             fontsize=14, fontweight='bold', y=1.01)

# (A) Accuracy comparison
ax1 = fig.add_subplot(2, 3, 1)
methods_s = svm_df['Method'].values
ax1.barh(methods_s, svm_df['Accuracy'].values * 100,
         color=['#5B7FA6']*7 + ['#E84040'], edgecolor='white', height=0.65)
ax1.set_title('(A) Accuracy (SVM-RBF, fair)'); ax1.set_xlim(0, 110); ax1.grid(axis='x', alpha=0.3)

# (B) Retrieval Bullseye
ax2 = fig.add_subplot(2, 3, 2)
ret_df_s = ret_df.sort_values('Bullseye', ascending=True)
ax2.barh(ret_df_s['Method'], ret_df_s['Bullseye'],
         color=['#E84040' if 'AMST' in n else '#5B7FA6' for n in ret_df_s['Method']])
ax2.set_title('(B) Retrieval Bullseye Rating (%)'); ax2.set_xlim(0, 105); ax2.grid(axis='x', alpha=0.3)

# (C) Ablation
ax3 = fig.add_subplot(2, 3, 3)
abl_accs = [r['Accuracy'] * 100 for r in abl_svm_results]
abl_names_p = [r['Method'][:35] for r in abl_svm_results]
ax3.barh(abl_names_p, abl_accs, color=['#AED6F1','#5DADE2','#2471A3','#1A5276','#E84040'])
ax3.set_title('(C) Ablation'); ax3.set_xlim(0, 105); ax3.grid(axis='x', alpha=0.3)

# (D) AMST gain with Bonferroni
ax4 = fig.add_subplot(2, 3, 4)
stat_df_s = stat_df.sort_values('Delta_pp', ascending=True)
colors_sig = ['#27AE60' if s else '#E74C3C' for s in stat_df_s['Significant_bonf']]
ax4.barh(stat_df_s['Baseline'], stat_df_s['Delta_pp'], color=colors_sig)
ax4.axvline(0, color='k', lw=1)
ax4.set_title('(D) AMST Gain (pp) | Green=Bonferroni-significant'); ax4.grid(axis='x', alpha=0.3)

# (E) Summary text
ax5 = fig.add_subplot(2, 3, 5); ax5.axis('off')
summary = (f'Dataset: MPEG-7 CE-Shape-1 ({len(contours)} images, {n_classes} classes)\n'
           f'AMST dim: {AMST_DIM}\n'
           f'5-fold CV (SVM-RBF, fair):\n'
           f'  AMST: {amst_acc_svm:.2f}%\n'
           f'  Best base: {best_base_svm:.2f}%\n'
           f'  Gain: +{amst_acc_svm-best_base_svm:.2f} pp\n'
           f'Stacking: {amst_acc_stack:.2f}%\n'
           f'Bonferroni sig. wins: {n_sig_bonf}/{N_COMPARISONS}\n'
           f'Retrieval Bullseye: {amst_bullseye:.1f}%')
ax5.text(0.05, 0.5, summary, fontsize=10, fontfamily='monospace', va='center',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# (F) Noise AUC
ax6 = fig.add_subplot(2, 3, 6)
noise_auc = {nm: np.trapz([v * 100 for v in noise_res[nm]], noise_levels[:4])
             for _, nm in noise_methods}
ax6.barh(list(noise_auc.keys()), list(noise_auc.values()),
         color=['#E84040' if 'AMST' in nm else '#5B7FA6' for nm in noise_auc])
ax6.set_title('(F) Noise Robustness AUC'); ax6.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/fig9_dashboard.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 9 saved.')

In [ ]:
# ── Save All Results ──────────────────────────────
import os
out_dir = '/content/amst_v11_results'
os.makedirs(out_dir, exist_ok=True)

svm_df[['Method','Accuracy','Accuracy_std','F1_macro','Precision_macro','Recall_macro','Dim']].to_csv(
    f'{out_dir}/classification_results.csv', index=False)
stat_df.to_csv(f'{out_dir}/significance_tests.csv', index=False)
pd.DataFrame(abl_svm_results).to_csv(f'{out_dir}/ablation.csv', index=False)
ret_df.to_csv(f'{out_dir}/retrieval_results.csv', index=False)
pd.DataFrame(noise_res, index=list(noise_levels[:4])).to_csv(f'{out_dir}/noise_robustness.csv')

print('Saved results to', out_dir)
for f in sorted(os.listdir(out_dir)):
    print(f'  {f}')

print('\n=== AMST v11 Complete — MPEG-7 CE-Shape-1 ===')
print('Dataset: 1,400 images, 70 classes')
print('Fair SVM comparison | Bonferroni correction | Bullseye rating | CNN baseline')